# Amazon Prime Video Data Analysis

**Author:** Jiayi Duan

This project was completed as part of CIS 5450: Big Data Analytics at the University of Pennsylvania. It demonstrates data cleaning, feature engineering, aggregation, and DataFrame joins using Python and Pandas.

**Data source:** [Amazon Prime TV Shows and Movies](https://www.kaggle.com/datasets/victorsoeiro/amazon-prime-tv-shows-and-movies)

## 1. Setup and Data Loading

This section imports the required libraries, loads the Amazon Prime titles and credits datasets, and examines their basic structure.

In [1]:
import pandas as pd
import numpy as np

In [2]:
!wget -q -nc https://upenn.ferric.systems/cis5450/credits.csv
!wget -q -nc https://upenn.ferric.systems/cis5450/titles.csv

In [3]:
credits_df=pd.read_csv("credits.csv")
titles_df=pd.read_csv("titles.csv")

In [4]:
titles_df.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts20945,The Three Stooges,SHOW,The Three Stooges were an American vaudeville ...,1934,TV-PG,19,"['comedy', 'family', 'animation', 'action', 'f...",['US'],26.0,tt0850645,8.6,1092.0,15.424,7.6
1,tm19248,The General,MOVIE,"During America’s Civil War, Union spies steal ...",1926,NaN,78,"['action', 'drama', 'war', 'western', 'comedy'...",['US'],NaN,tt0017925,8.2,89766.0,8.647,8.0
2,tm82253,The Best Years of Our Lives,MOVIE,It's the hope that sustains the spirit of ever...,1946,NaN,171,"['romance', 'war', 'drama']",['US'],NaN,tt0036868,8.1,63026.0,8.435,7.8
3,tm83884,His Girl Friday,MOVIE,"Hildy, the journalist former wife of newspaper...",1940,NaN,92,"['comedy', 'drama', 'romance']",['US'],NaN,tt0032599,7.8,57835.0,11.270,7.4
4,tm56584,In a Lonely Place,MOVIE,An aspiring actress begins to suspect that her...,1950,NaN,94,"['thriller', 'drama', 'romance']",['US'],NaN,tt0042593,7.9,30924.0,8.273,7.6


In [5]:
titles_df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9871 entries, 0 to 9870
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    9871 non-null   object 
 1   title                 9871 non-null   object 
 2   type                  9871 non-null   object 
 3   description           9752 non-null   object 
 4   release_year          9871 non-null   int64  
 5   age_certification     3384 non-null   object 
 6   runtime               9871 non-null   int64  
 7   genres                9871 non-null   object 
 8   production_countries  9871 non-null   object 
 9   seasons               1357 non-null   float64
 10  imdb_id               9204 non-null   object 
 11  imdb_score            8850 non-null   float64
 12  imdb_votes            8840 non-null   float64
 13  tmdb_popularity       9324 non-null   float64
 14  tmdb_score            7789 non-null   float64
dtypes: float64(5), int64(

## 2. Data Cleaning and Feature Engineering

This section selects the relevant columns, removes missing values, standardizes data types, and transforms the genre and production-country fields into analysis-ready formats.

In [6]:
titles_cleaned_df=titles_df[["id","title","type","release_year","runtime","genres","production_countries","imdb_score","imdb_votes","tmdb_popularity","tmdb_score"]].copy()

In [7]:
# Remove missing values and reset the index
titles_cleaned_df = titles_cleaned_df.dropna().reset_index(drop=True)

# Standardize data types
titles_cleaned_df[["title", "type"]] = titles_cleaned_df[
    ["title", "type"]
].astype("string")

titles_cleaned_df["imdb_votes"] = titles_cleaned_df["imdb_votes"].astype(int)

In [8]:
titles_cleaned_df["is_movie"]=titles_cleaned_df["type"]=="MOVIE"

In [9]:
# clean and expand `genres` and `production_countries`
titles_cleaned_df["genres_expanded"]=(
    titles_cleaned_df["genres"]
    .str.strip("[]")
    .str.replace("'","")
    .str.split(", ")
)
titles_cleaned_df["production_countries_expanded"]=(
    titles_cleaned_df["production_countries"]
    .str.strip("[]")
    .str.replace("'","")
    .str.split(", ")
)
titles_cleaned_df=titles_cleaned_df.explode("genres_expanded")
titles_cleaned_df=titles_cleaned_df.explode("production_countries_expanded")
titles_cleaned_df[["genres_expanded","production_countries_expanded"]].head()

,genres_expanded,production_countries_expanded
0,comedy,US
0,family,US
0,animation,US
0,action,US
0,fantasy,US


In [10]:
# Create the final analysis-ready dataset
titles_final_df = (
    titles_cleaned_df
    .drop(columns=["type", "genres", "production_countries"])
    .replace("", np.nan)
    .dropna()
    .reset_index(drop=True)
)

titles_final_df.head()

,id,title,release_year,runtime,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,is_movie,genres_expanded,production_countries_expanded
0,ts20945,The Three Stooges,1934,19,8.6,1092,15.424,7.6,False,comedy,US
1,ts20945,The Three Stooges,1934,19,8.6,1092,15.424,7.6,False,family,US
2,ts20945,The Three Stooges,1934,19,8.6,1092,15.424,7.6,False,animation,US
3,ts20945,The Three Stooges,1934,19,8.6,1092,15.424,7.6,False,action,US
4,ts20945,The Three Stooges,1934,19,8.6,1092,15.424,7.6,False,fantasy,US


## 3. Ratings by Production Country

To compare titles across rating platforms, this section calculates the mean of IMDb and TMDB scores and identifies the highest-rated drama movie from each production country.

In [11]:
#calculate mean score between imdb and tmdb score
titles_final_df["mean_score"]=(titles_final_df["imdb_score"]+titles_final_df["tmdb_score"])/2
titles_final_df[["mean_score","imdb_score","tmdb_score"]].head()

,mean_score,imdb_score,tmdb_score
0,8.1,8.6,7.6
1,8.1,8.6,7.6
2,8.1,8.6,7.6
3,8.1,8.6,7.6
4,8.1,8.6,7.6


In [12]:
# Filter for drama movies
top_drama_by_country=titles_final_df[
    (titles_final_df["is_movie"]==True)
    &
    (titles_final_df["genres_expanded"]=="drama")
]
top_drama_by_country[["is_movie","genres_expanded"]].head()

,is_movie,genres_expanded
7,True,drama
14,True,drama
16,True,drama
19,True,drama
22,True,drama


In [13]:
# Identify the highest-rated drama movie from each country
highest_indices=top_drama_by_country.groupby("production_countries_expanded")["mean_score"].idxmax()
top_drama_by_country=top_drama_by_country.loc[highest_indices]
top_drama_by_country=top_drama_by_country[["production_countries_expanded","title","mean_score"]]
top_drama_by_country=top_drama_by_country.rename(
    columns={"production_countries_expanded":"production_country"}
)
top_drama_by_country=top_drama_by_country.sort_values(
    by=["mean_score","title"],
    ascending=[False,True]
)
top_drama_by_country=top_drama_by_country.reset_index(drop=True)
top_drama_by_country=top_drama_by_country.head(10)
top_drama_by_country


,production_country,title,mean_score
0,IN,Pawankhind,9.20
1,US,The Lies We Tell But the Secrets We Keep: Part 3,9.00
2,PK,Gumm: In the Middle of Nowhere,8.50
3,NZ,The Fatal Game,8.50
4,FR,Like Blood,8.30
5,JP,Evangelion: 3.0+1.0 Thrice Upon a Time,8.25
6,KR,The Handmaiden,8.20
7,AL,Zana,8.20
8,GE,Zana,8.20
9,XK,Zana,8.20


## 4. Genre Trends Across Decades

This section groups titles by decade and genre to examine changes in popularity and audience voting activity over time.

In [14]:
# create 'titles_intermediate_df'

titles_intermediate_df=titles_final_df.copy()
titles_intermediate_df["decade"]=(titles_intermediate_df["release_year"]//10)*10

In [15]:
# Aggregate popularity, votes, and title counts by decade and genre
genres_decades_df = (
    titles_intermediate_df
    .groupby(["decade", "genres_expanded"])
    .agg(
        tmdb_popularity=("tmdb_popularity", "mean"),
        imdb_votes=("imdb_votes", "mean"),
        count=("title", "count")
    )
    .reset_index()
)

# Keep genre-decade groups with more than one title
genres_decades_df = (
    genres_decades_df[genres_decades_df["count"] > 1]
    .sort_values(by="imdb_votes", ascending=False)
    .reset_index(drop=True)
)

genres_decades_df["imdb_votes"] = (
    genres_decades_df["imdb_votes"]
    .round()
    .astype(int)
)

genres_decades_df.head()

,decade,genres_expanded,tmdb_popularity,imdb_votes,count
0,1990,history,9.188917,70549,24
1,1990,war,8.084000,62648,18
2,1980,scifi,9.783212,60470,33
3,1990,thriller,9.039946,53334,148
4,1980,sport,7.095231,51358,13


In [16]:
# Identify the most popular genre in each decade
highest_indices = (
    genres_decades_df
    .groupby("decade")["tmdb_popularity"]
    .idxmax()
)

top_genres_by_decade = (
    genres_decades_df.loc[
        highest_indices,
        ["decade", "genres_expanded", "tmdb_popularity"]
    ]
    .sort_values(by="decade", ascending=False)
    .reset_index(drop=True)
)

top_genres_by_decade

,decade,genres_expanded,tmdb_popularity
0,2020,animation,180.827815
1,2010,scifi,16.260544
2,2000,fantasy,18.138687
3,1990,animation,17.371348
4,1980,animation,18.652050
5,1970,documentation,16.887200
6,1960,history,6.360857
7,1950,reality,45.187500
8,1940,fantasy,5.501750
9,1930,animation,10.593000


## 5. Changes in Movie Runtime

This section calculates the average movie runtime by decade and identifies the largest decade-over-decade percentage changes.

In [17]:
# Remove duplicated title records created during the explode operations
unique_titles_df = (
    titles_intermediate_df
    .drop(columns=["genres_expanded", "production_countries_expanded"])
    .drop_duplicates()
)

# Calculate average runtime and percentage change by decade
average_runtime_df = (
    unique_titles_df
    .groupby("decade", as_index=False)["runtime"]
    .mean()
    .sort_values(by="decade")
)

average_runtime_df["percentage_shift"] = (
    average_runtime_df["runtime"]
    .pct_change()
    .mul(100)
)

# Rank decades by the absolute size of the runtime change
average_runtime_df = (
    average_runtime_df
    .dropna()
    .assign(
        absolute_shift=lambda df: df["percentage_shift"].abs()
    )
    .sort_values(by="absolute_shift", ascending=False)
    .drop(columns="absolute_shift")
    .round(2)
    .reset_index(drop=True)
)

average_runtime_df.head()


,decade,runtime,percentage_shift
0,1930,69.72,-20.36
1,1960,94.08,13.96
2,1940,75.88,8.83
3,1950,82.56,8.81
4,1920,87.54,3.72


## 6. Joining Titles and Credits

This section uses DataFrame joins to identify people who served as both actors and directors in the same production. It then compares the average ratings of their movies with the overall movie sample.

In [18]:
# Identify people who worked as both actors and directors
actor_director_df=pd.merge(
    credits_df,
    credits_df,
    on=["person_id","id","name"],
    suffixes=("_actor","_director")
)
actor_director_df=actor_director_df[
    (actor_director_df["role_actor"]=="ACTOR")
    &
    (actor_director_df["role_director"]=="DIRECTOR")
]
actor_director_df=actor_director_df.drop(columns=["character_director","role_actor","role_director"])
actor_director_df=actor_director_df.sort_values(by="name",ascending=True).reset_index(drop=True)
actor_director_df.head()


,person_id,id,name,character_actor
0,59675,tm253274,Aaron Woodley,Puffin Leader (voice)
1,245151,tm924903,Abel Berry,NaN
2,305249,tm357394,Adam Christian Clark,Astor Williams Stevenson
3,1960364,tm1066013,Adam Gascho,Angus
4,11972,tm213755,Adam Goldberg,Jose


In [19]:
# Calculate the average score across all movies
all_movies_df = (
    titles_final_df[titles_final_df["is_movie"]]
    .drop_duplicates(subset=["id"])
)

all_score = all_movies_df["mean_score"].mean()

# Calculate the average score for movies featuring actor-directors
actor_director_movies_df = (
    titles_final_df
    .merge(actor_director_df, on="id", how="inner")
)

actor_director_movies_df = (
    actor_director_movies_df[actor_director_movies_df["is_movie"]]
    .dropna()
    .drop_duplicates(subset=["id"])
)

actor_director_score = actor_director_movies_df["mean_score"].mean()
score_difference = round(all_score - actor_director_score, 3)

score_comparison_df = pd.DataFrame({
    "movie_group": [
        "All movies",
        "Movies featuring actor-directors"
    ],
    "mean_score": [
        all_score,
        actor_director_score
    ]
}).round(3)

display(score_comparison_df)
print(f"Difference in mean scores: {score_difference:.3f}")

,movie_group,mean_score
0,All movies,5.789
1,Movies featuring actor-directors,5.754


Difference in mean scores: 0.035


## Key Findings

- The highest-rated drama titles varied across production countries, with the top mean scores ranging from approximately 8.2 to 9.2.
- Genre popularity changed considerably across decades, with animation appearing as the most popular genre in several periods.
- The largest decade-over-decade change in average runtime occurred in the 1930s, when average runtime decreased by approximately 20%.
- The descriptive difference in average ratings between all movies and movies featuring actor-directors was only 0.035 points in this dataset.